# 7.8 — Clustering a Document Corpus

**Chapter 7, section 7.5.4.**

**The question this notebook answers:** what does it take to read a clustering back as words?

Five stages: the four-stage TF–IDF pipeline of notebook 7.6, with a clustering estimator appended.
Fitting it produces one object holding the vocabulary, the inverse document frequencies **and** the
cluster centroids — and that is what makes the last step possible. A centroid is a point in the
feature space, so its largest coordinates can be looked up in the vocabulary that defined that
space, turning an anonymous vector into a list of words.

The notebook also settles two choices the chapter makes and does not measure: why cosine distance
rather than Euclidean, and what the silhouette coefficient can and cannot tell you about a corpus
of film reviews.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-7.8")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. The corpus and the four featurization stages

The same hashed quarter of the film reviews as notebooks 7.6 and 7.7, cleaned the same way. The
vectorizer uses the document-frequency thresholds of 7.7's run C: without them the top of the
vocabulary is *movie*, *film*, *one*, *like*, which are in half the corpus and therefore separate
nothing.

The corpus is cached because the pipeline has three estimator stages, each of which is a pass.

In [2]:
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF

reviews = spark.read.csv(os.path.join(DATA, "IMDB Dataset.csv.bz2"), header=True)

cleaned = F.regexp_replace(F.col("text"), r'<[^>]+>', "")
cleaned = F.regexp_replace(cleaned, r"[^a-zA-Z ]", "")
cleaned = F.regexp_replace(cleaned, r"\b\w{1,2}\b", "")
cleaned = F.trim(F.regexp_replace(cleaned, r"\s+", " "))

train_docs = (reviews
              .withColumn("bucket", F.crc32(F.col("text")) % 100)
              .filter("bucket < 25")
              .withColumn("text_c", F.lower(cleaned))
              .select("text_c")
              .cache())

print("documents:", train_docs.count())

documents: 12520


In [3]:
tokenizer = Tokenizer(inputCol="text_c", outputCol="words")
remover = StopWordsRemover(inputCol=tokenizer.getOutputCol(), outputCol="filtered")
countVectorizer = CountVectorizer(inputCol=remover.getOutputCol(), outputCol="rawFeatures",
                                  vocabSize=1000, minDF=10.0, maxDF=5000.0)
idf = IDF(inputCol=countVectorizer.getOutputCol(), outputCol="featuresIDF")

featurizer = Pipeline(stages=[tokenizer, remover, countVectorizer, idf]).fit(train_docs)
vocabulary = featurizer.stages[2].vocabulary

features = featurizer.transform(train_docs).select("filtered", "featuresIDF").cache()
print("documents featurized:", features.count())
print("vocabulary starts:", vocabulary[:10])

documents featurized: 12520
vocabulary starts: ['good', 'even', 'time', 'really', 'see', 'story', 'much', 'well', 'get', 'people']


---

## 2. Choosing $k$ by silhouette

`ClusteringEvaluator` returns the silhouette coefficient, which lies in $[-1, 1]$: one means every
point is far closer to its own cluster's members than to any other cluster's, zero means the
clusters overlap so much that the assignment is arbitrary, and negative means points are on
average closer to a cluster they were not put in.

The evaluator's `distanceMeasure` must match the clusterer's. Scoring a cosine clustering with the
default squared-Euclidean silhouette measures a different geometry from the one that was
optimized.

In [4]:
import time
from pyspark.ml.clustering import KMeans, BisectingKMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(featuresCol="featuresIDF", distanceMeasure="cosine")
ks = [2, 3, 4, 5, 6, 8]

sweep = {}
for cls in (KMeans, BisectingKMeans):
    row = []
    for k in ks:
        model = cls(featuresCol="featuresIDF", k=k, distanceMeasure="cosine", seed=1).fit(features)
        row.append(evaluator.evaluate(model.transform(features)))
    sweep[cls.__name__] = row

print(f"{'k':>4}" + "".join(f"{k:>12}" for k in ks))
for name, row in sweep.items():
    print(f"{name:<16}" + "".join(f"{s:>12.4f}" for s in row))

   k           2           3           4           5           6           8
KMeans                0.0078      0.0063      0.0064      0.0089      0.0095      0.0095
BisectingKMeans       0.0080      0.0064      0.0062      0.0049      0.0045      0.0057


Every one of those numbers is within a hundredth of zero.

That is the honest result for this corpus, and it is worth stating plainly rather than picking the
largest and moving on. Fifty thousand film reviews are not six topics; they are fifty thousand
opinions about different films, written in one register, drawn from one vocabulary. There is no
well-separated cluster structure for the silhouette to find, and a difference between 0.008 and
0.010 is not evidence that one $k$ is better than another.

**What follows from that.** The silhouette has done its job by telling us it cannot choose. The
number of clusters becomes a decision about what is useful to look at rather than one the data
settles, and the chapter's $k = 6$ is as defensible as any. What the clustering still produces —
and what the rest of this notebook is about — is a partition whose centroids are readable, which
is useful for exploring a corpus even when the partition is not a discovery about it.

In [5]:
# For contrast: the same evaluator on data that does have cluster structure.
from pyspark.ml.feature import VectorAssembler

blobs = (spark.range(600)
         .withColumn("g", (F.col("id") % 3).cast("double"))
         .withColumn("x", F.col("g") * 10 + F.randn(seed=1))
         .withColumn("y", F.col("g") * 10 + F.randn(seed=2)))
blobs = VectorAssembler(inputCols=["x", "y"], outputCol="featuresIDF").transform(blobs)

m = KMeans(featuresCol="featuresIDF", k=3, seed=1).fit(blobs)
plain = ClusteringEvaluator(featuresCol="featuresIDF")
print(f"three well-separated blobs, k=3: silhouette {plain.evaluate(m.transform(blobs)):.4f}")
print("the film reviews, at every k tried:  about 0.01")

three well-separated blobs, k=3: silhouette 0.9819
the film reviews, at every k tried:  about 0.01


---

## 3. The five-stage pipeline

The clusterer is appended to the four featurization stages rather than fitted separately. The
chapter gives the reason and section 5 demonstrates it: keeping them together is what keeps the
centroids interpretable after the model is saved.

In [6]:
kmeans = BisectingKMeans(featuresCol="featuresIDF", k=6,
                         distanceMeasure="cosine", seed=1)

pipeline = Pipeline(stages=[tokenizer, remover, countVectorizer, idf, kmeans])
model = pipeline.fit(train_docs)

clustered = model.transform(train_docs).withColumn("length", F.size("filtered")).cache()
silhouette = evaluator.evaluate(clustered)
print(f"silhouette (cosine): {silhouette:.4f}")

model.stages

silhouette (cosine): 0.0045


[Tokenizer_1a9267fd2af9,
 StopWordsRemover_d60b213a25a6,
 CountVectorizerModel: uid=CountVectorizer_aa95d5a675ae, vocabularySize=1000,
 IDFModel: uid=IDF_ffce33883bbd, numDocs=12520, numFeatures=1000,
 BisectingKMeansModel: uid=BisectingKMeans_92ae6685b791, k=6, distanceMeasure=cosine, numFeatures=1000]

In [7]:
clustered.groupBy("prediction").agg(
    F.count("*").alias("documents"),
    F.round(F.avg("length"), 1).alias("mean_words")).orderBy("prediction").show()

+----------+---------+----------+
|prediction|documents|mean_words|
+----------+---------+----------+
|         0|     1022|      88.4|
|         1|     2433|      77.6|
|         2|     2651|     124.4|
|         3|     2781|      96.5|
|         4|     1452|     150.1|
|         5|     2181|     163.8|
+----------+---------+----------+



---

## 4. Reading the centroids back as words

Each centroid is a point in the thousand-dimensional feature space. Its largest coordinates are
the terms that most define the cluster, and the vocabulary — sitting in stage 2 of the same fitted
object — says which term each coordinate is.

This loop is the payload of the notebook. Without it the output of the clustering is a column of
integers.

In [8]:
vocabulary = model.stages[2].vocabulary

for i, centre in enumerate(model.stages[4].clusterCenters()):
    top = sorted(range(len(centre)), key=lambda j: -centre[j])[:8]
    print(i, [vocabulary[j] for j in top])

0 ['show', 'worst', 'ever', 'episode', 'seen', 'series', 'ive', 'funny']
1 ['bad', 'dont', 'good', 'movies', 'watch', 'acting', 'really', 'think']
2 ['really', 'horror', 'get', 'even', 'bad', 'plot', 'people', 'dont']
3 ['great', 'story', 'love', 'best', 'excellent', 'well', 'war', 'life']
4 ['young', 'man', 'father', 'life', 'woman', 'wife', 'mother', 'family']
5 ['films', 'character', 'two', 'story', 'also', 'characters', 'much', 'well']


In [9]:
# The shortest review in each cluster, as a spot check rather than proof. A single document
# is not evidence about a cluster of two thousand; it is a way of seeing what the words above
# are describing.
for i in range(kmeans.getK()):
    text = (clustered.filter(F.col("prediction") == i)
            .orderBy(F.length("text_c"))
            .select("text_c").first()["text_c"])
    print(f"--- cluster {i} ---")
    print(" ", text[:200])
    print()

--- cluster 0 ---
  hated with all being worst movie ever mentally scarred help was that badtrust



--- cluster 1 ---
  rating does not begin express how dull depressing and relentlessly bad this movie

--- cluster 2 ---
  unwatchable you cant even make past the first three minutes and this coming from huge adam sandler fan

--- cluster 3 ---
  read the book forget the movie

--- cluster 4 ---
  this movies really special its about young french who barcelona spain order study and barcleona meets other youngs europeean like him this film hes the european movies the year watch



--- cluster 5 ---
  one favorite scenes the beginning when guests private yacht decide take impromptu swim their underwear rather risqu for



---

## 5. Why cosine and not Euclidean

The chapter's reason is that document similarity is about the *proportions* of words rather than
the count of them, so a long review and a short review about the same film should be close. Under
Euclidean distance they are not, because a long document has larger coordinates everywhere and is
therefore far from a short one no matter what either says.

That claim is measurable. Refit the same clusterer with the default squared-Euclidean distance and
look at what the clusters have in common.

In [10]:
euclidean = Pipeline(stages=[tokenizer, remover, countVectorizer, idf,
                             BisectingKMeans(featuresCol="featuresIDF", k=6, seed=1)]).fit(train_docs)

euclidean_clusters = (euclidean.transform(train_docs)
                      .withColumn("length", F.size("filtered")))

euclidean_clusters.groupBy("prediction").agg(
    F.count("*").alias("documents"),
    F.round(F.avg("length"), 1).alias("mean_words")).orderBy("mean_words").show()

+----------+---------+----------+
|prediction|documents|mean_words|
+----------+---------+----------+
|         0|     3936|      61.3|
|         1|     2590|      65.9|
|         2|     2030|     109.9|
|         3|     1861|     133.6|
|         4|     1601|     230.0|
|         5|      502|     399.6|
+----------+---------+----------+



In [11]:
def length_spread(df):
    stats = (df.groupBy("prediction").agg(F.avg("length").alias("m"))
             .agg(F.min("m").alias("lo"), F.max("m").alias("hi")).first())
    return stats["lo"], stats["hi"], stats["hi"] / stats["lo"]

for label, df in (("cosine   ", clustered), ("euclidean", euclidean_clusters)):
    lo, hi, ratio = length_spread(df)
    print(f"{label}  cluster mean length from {lo:6.1f} to {hi:6.1f} words   "
          f"a factor of {ratio:.1f}")

cosine     cluster mean length from   77.6 to  163.8 words   a factor of 2.1


euclidean  cluster mean length from   61.3 to  399.6 words   a factor of 6.5


The Euclidean clustering has sorted the corpus by length. Its clusters run from the shortest
reviews to the longest in an almost unbroken ladder, which is a fact about how much people wrote
rather than about what they wrote. The cosine clustering still varies in length — longer reviews
do use different words — but by a much smaller factor, and its clusters divide on subject and
sentiment instead.

In [12]:
# What the Euclidean centroids look like as words. Compare them with the cosine ones: the same
# generic terms recur across several clusters, because what separates these clusters is the
# magnitude of the vectors rather than which terms are in them.
euclid_vocab = euclidean.stages[2].vocabulary
for i, centre in enumerate(euclidean.stages[4].clusterCenters()):
    top = sorted(range(len(centre)), key=lambda j: -centre[j])[:8]
    print(i, [euclid_vocab[j] for j in top])

0 ['great', 'story', 'good', 'best', 'love', 'well', 'films', 'life']
1 ['bad', 'watch', 'seen', 'movies', 'ever', 'dont', 'see', 'show']
2 ['bad', 'show', 'really', 'dont', 'people', 'movies', 'even', 'good']
3 ['story', 'love', 'life', 'also', 'well', 'films', 'great', 'time']
4 ['even', 'character', 'also', 'people', 'really', 'films', 'much', 'time']
5 ['scene', 'films', 'get', 'even', 'hes', 'also', 'two', 'know']


---

## 6. Why the clusterer belongs inside the pipeline

Save the whole thing, reload it in one call, and the centroids are still readable — because the
vocabulary that indexes them was saved alongside them, in the same object.

In [13]:
import shutil
from pyspark.ml import PipelineModel

path = os.path.join(SCRATCH, "doc_clustering_model")
shutil.rmtree(path, ignore_errors=True)
model.save(path)

reloaded = PipelineModel.load(path)
reloaded_vocab = reloaded.stages[2].vocabulary

for i, centre in enumerate(reloaded.stages[4].clusterCenters()):
    top = sorted(range(len(centre)), key=lambda j: -centre[j])[:8]
    print(i, [reloaded_vocab[j] for j in top])

0 ['show', 'worst', 'ever', 'episode', 'seen', 'series', 'ive', 'funny']
1 ['bad', 'dont', 'good', 'movies', 'watch', 'acting', 'really', 'think']
2 ['really', 'horror', 'get', 'even', 'bad', 'plot', 'people', 'dont']
3 ['great', 'story', 'love', 'best', 'excellent', 'well', 'war', 'life']
4 ['young', 'man', 'father', 'life', 'woman', 'wife', 'mother', 'family']
5 ['films', 'character', 'two', 'story', 'also', 'characters', 'much', 'well']


In [14]:
assert reloaded_vocab == vocabulary
assert [list(c) for c in reloaded.stages[4].clusterCenters()] == \
       [list(c) for c in model.stages[4].clusterCenters()]
print("vocabulary and centroids survived the round trip unchanged")

vocabulary and centroids survived the round trip unchanged


In [15]:
# And a new document is scored through the whole workflow in one call.
new_docs = spark.createDataFrame(
    [("a quiet family drama about a father and his young son",),
     ("worst horror movie ever made the acting was terrible",)],
    ["text_c"])

reloaded.transform(new_docs).select("text_c", "prediction").show(truncate=55)

+-----------------------------------------------------+----------+
|                                               text_c|prediction|
+-----------------------------------------------------+----------+
|a quiet family drama about a father and his young son|         4|
| worst horror movie ever made the acting was terrible|         0|
+-----------------------------------------------------+----------+



Had the clustering model been saved on its own, the reload would have produced six vectors of a
thousand numbers with nothing to say which coordinate is which word, and a new document could not
have been featurized the same way without rebuilding the vectorizer by hand.

---

## What to take away

The five-stage pipeline holds the vocabulary, the inverse document frequencies and the cluster
centroids in one object, and the last section is possible only because of that.

Measured on the hashed quarter of the corpus, **12,520 documents** and a 1,000-term vocabulary
with `minDF=10, maxDF=5000`:

* The **silhouette is about 0.01 at every $k$ tried**, for both K-means and bisecting K-means,
  under cosine distance. Three well-separated synthetic blobs score above 0.9 with the same
  evaluator. The metric is saying that this corpus has no cluster structure to find, and the
  honest reading is to treat $k$ as a display choice rather than a discovery.
* The centroids are nonetheless **readable and distinct**: television and episodes, negative
  reviews, horror, praise, family drama, and general film criticism.
* Under **squared-Euclidean** distance the same clusterer sorts the corpus by length, with mean
  cluster lengths spanning a factor of six. Under **cosine** the same span is about two. This is
  the chapter's argument for cosine, measured.

In [16]:
spark.stop()